# Teacher Fine-Tuning: DeBERTa-v3-large

**Phase 3 of CanaryOS Text Classification Research**

Fine-tunes `microsoft/deberta-v3-large` (435M params) on the Phase 1 synthetic dataset
with a dual-head architecture:
- **Binary head:** scam/safe classification (hard gate: F1 > 0.80 on real-world holdout)
- **Intent head:** 8 sigmoid outputs (urgency, authority, financial_request, remote_access, reward_lottery, impersonation, romance_grooming, crypto)

The teacher runs server-side only (Colab) and never deploys to device. Its purpose is to
establish an accuracy ceiling and produce calibrated soft labels for Phase 4 distillation
into MobileBERT.

**Deliverables:**
1. Teacher checkpoint passing F1 gates (>0.95 synthetic, >0.80 holdout)
2. ECE calibration measurement before/after temperature scaling
3. Pre-computed soft labels at T={2,3,4,5} for Phase 4 distillation

**Environment:** Google Colab (T4 default, A100 migration guide included)

**Data:** Phase 1 synthetic dataset (~18,353 training samples, 8 scam vectors + safe)

In [ ]:
# =============================================================================
# Cell 1: Environment Setup + Drive Mount (D-01, D-02)
# =============================================================================
# Install required packages (DeBERTa-v3 uses SentencePiece tokenizer, not WordPiece)
!pip install -q transformers>=4.48.0 datasets>=3.0.0 accelerate>=0.25.0 \
    evaluate>=0.4.0 torchmetrics>=1.2.0 sentencepiece

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")
    print("Go to Runtime > Change runtime type > GPU (T4 recommended)")

# Mount Google Drive for checkpoint persistence (D-04)
# Colab sessions disconnect after ~90 min idle or 12 hours total.
# All checkpoints are saved to Drive so training can resume.
from google.colab import drive
drive.mount("/content/drive")

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/canaryos_teacher"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

In [ ]:
# =============================================================================
# Cell 2: Configuration (D-02, D-03, D-12)
# =============================================================================
# DeBERTa-v3-large fits on T4 (16GB VRAM) with batch size 4, gradient
# accumulation 4, FP16, and gradient checkpointing. Effective batch = 16.

# === T4 Configuration (Default -- Free Colab) ===
T4_CONFIG = {
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,      # effective batch = 16
    "learning_rate": 8e-6,                 # DeBERTa-v3-large sweet spot: 5e-6 to 1e-5
    "num_train_epochs": 3,                 # 2-3 epochs to avoid overfitting
    "warmup_ratio": 0.1,                   # ~10% warmup steps
    "weight_decay": 0.01,                  # standard for DeBERTa
    "lr_scheduler_type": "cosine",         # cosine decay recommended
    "fp16": True,                          # required for T4 memory
    "gradient_checkpointing": True,        # required for T4 memory
    "max_grad_norm": 1.0,                  # gradient clipping
    "max_seq_length": 128,                 # scam texts are short
}

# === A100 Configuration (Colab Pro Migration -- D-03) ===
# Same effective batch size (16) but larger per-device batch.
# No gradient checkpointing needed with 40GB VRAM.
A100_CONFIG = {
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,      # effective batch = 16 (same)
    "learning_rate": 8e-6,                 # unchanged
    "num_train_epochs": 3,                 # unchanged
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "fp16": True,                          # A100 also supports bf16
    "gradient_checkpointing": False,       # not needed with 40GB
    "max_grad_norm": 1.0,
    "max_seq_length": 128,
}

# === Retry Configs (D-12: If holdout F1 < 0.80 after first attempt) ===
# Retry 1: Lower learning rate + more epochs (conservative approach)
RETRY_1 = {
    **T4_CONFIG,
    "learning_rate": 3e-6,        # halved from default
    "num_train_epochs": 5,        # more epochs at lower LR
    "warmup_ratio": 0.15,         # slightly longer warmup
}

# Retry 2: Adjusted LR + class weighting flag
# Apply class weights in loss function: training data has ~8,230 scam vs ~10,123 safe
# so scam is the minority in training despite being majority in holdout (108/202)
RETRY_2 = {
    **T4_CONFIG,
    "learning_rate": 6e-6,        # between default and retry 1
    "num_train_epochs": 4,
    "use_class_weights": True,    # flag: apply class weights in loss computation
}

# ---- Active Configuration Selector ----
# Change this to switch configs:
#   ACTIVE_CONFIG = T4_CONFIG    (default, free Colab T4)
#   ACTIVE_CONFIG = A100_CONFIG  (Colab Pro A100)
#   ACTIVE_CONFIG = RETRY_1     (first retry if holdout F1 < 0.80)
#   ACTIVE_CONFIG = RETRY_2     (second retry with class weighting)
ACTIVE_CONFIG = T4_CONFIG

# Paths
MODEL_NAME = "microsoft/deberta-v3-large"
DATA_DIR = "research/data"  # relative path within Colab working directory
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/canaryos_teacher"

print("Active configuration:")
for k, v in ACTIVE_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nModel: {MODEL_NAME}")
print(f"Data dir: {DATA_DIR}")
print(f"Drive checkpoint dir: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# =============================================================================
# Cell 3: Data Loading + Intent Label Mapping (D-05)
# =============================================================================
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from collections import Counter

# 8 intent labels for the multi-label sigmoid head
INTENT_LABELS = [
    "urgency", "authority", "financial_request", "remote_access",
    "reward_lottery", "impersonation", "romance_grooming", "crypto"
]

# Mapping from scam vector to 8-dim binary intent labels
# Each vector activates the intents it typically exhibits.
# This mapping is approximate (D-05, Pitfall 6 in RESEARCH.md).
# Intent head is informational only -- does NOT block Phase 4 (D-06).
VECTOR_TO_INTENTS = {
    "crypto_investment":        [0, 0, 1, 0, 0, 0, 0, 1],  # financial_request + crypto
    "romance_grooming":         [0, 0, 1, 0, 0, 0, 1, 0],  # financial_request + romance
    "tech_support":             [1, 1, 0, 1, 0, 0, 0, 0],  # urgency + authority + remote_access
    "government_impersonation": [1, 1, 1, 0, 0, 1, 0, 0],  # urgency + authority + financial + impersonation
    "lottery_reward":           [0, 0, 1, 0, 1, 0, 0, 0],  # financial_request + reward_lottery
    "phishing":                 [1, 1, 0, 0, 0, 1, 0, 0],  # urgency + authority + impersonation
    "remote_access":            [1, 0, 0, 1, 0, 0, 0, 0],  # urgency + remote_access
    "urgency_payment":          [1, 0, 1, 0, 0, 0, 0, 0],  # urgency + financial_request
    "safe":                     [0, 0, 0, 0, 0, 0, 0, 0],  # all zeros
}

# Load DeBERTa-v3-large tokenizer (SentencePiece, 128K vocab)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {type(tokenizer).__name__}, vocab size: {tokenizer.vocab_size}")


class ScamDataset(Dataset):
    """Dataset class for scam/safe text classification with intent labels.

    Loads JSONL files with fields: text, label, vector, [split].
    Returns tokenized inputs with binary labels and 8-dim intent vectors.
    """

    def __init__(self, jsonl_path, tokenizer, max_length=128, split=None):
        self.samples = []
        with open(jsonl_path) as f:
            for line in f:
                sample = json.loads(line)
                # Filter by split field if provided (holdout has no split field)
                if split is None or sample.get("split") == split:
                    self.samples.append(sample)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        encoding = self.tokenizer(
            sample["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Binary label: scam=1, safe=0
        label = 1 if sample["label"] == "scam" else 0

        # Intent labels: 8-dim binary vector from vector field
        vector = sample.get("vector", "safe")
        intent = VECTOR_TO_INTENTS.get(vector, [0] * 8)

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
            "intent_labels": torch.tensor(intent, dtype=torch.float),
        }


# Load datasets
max_len = ACTIVE_CONFIG["max_seq_length"]

# Synthetic data: has split field (train/val)
synthetic_path = os.path.join(DATA_DIR, "synthetic_scam_v1.jsonl")
train_dataset = ScamDataset(synthetic_path, tokenizer, max_length=max_len, split="train")
val_dataset = ScamDataset(synthetic_path, tokenizer, max_length=max_len, split="val")

# Synthetic test: separate file, no split filtering needed
test_path = os.path.join(DATA_DIR, "test_split.jsonl")
test_dataset = ScamDataset(test_path, tokenizer, max_length=max_len, split=None)

# Real-world holdout: no split field, load all rows
holdout_path = os.path.join(DATA_DIR, "holdout_realworld.jsonl")
holdout_dataset = ScamDataset(holdout_path, tokenizer, max_length=max_len, split=None)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")
print(f"  Holdout: {len(holdout_dataset)} samples")

# Print per-vector distribution for training set
train_vectors = Counter(s["vector"] for s in train_dataset.samples)
train_labels = Counter(s["label"] for s in train_dataset.samples)
print(f"\nTraining set label distribution:")
for label, count in sorted(train_labels.items()):
    print(f"  {label}: {count}")
print(f"\nTraining set per-vector distribution:")
for vector, count in sorted(train_vectors.items(), key=lambda x: -x[1]):
    print(f"  {vector}: {count}")

In [ ]:
# =============================================================================
# Cell 4: Model Definition -- DualHeadDeBERTaTeacher (D-05)
# =============================================================================
# Custom dual-head model wrapping DebertaV2Model (not ForSequenceClassification)
# because we need both binary and multi-label heads on a shared encoder.

import torch.nn as nn
from transformers import DebertaV2Model, DebertaV2PreTrainedModel


class DualHeadDeBERTaTeacher(DebertaV2PreTrainedModel):
    """DeBERTa-v3-large with dual classification heads.

    Architecture:
        DebertaV2Model (shared encoder)
            -> [CLS] token -> Dropout
                -> binary_head: Linear(hidden_size, 2)  -- scam/safe
                -> intent_head: Linear(hidden_size, 8)  -- 8 sigmoid intents

    Loss:
        combined = 0.7 * CrossEntropyLoss(binary) + 0.3 * BCEWithLogitsLoss(intent)
        Binary head is weighted more heavily since it is the hard gate (D-06).
    """

    def __init__(self, config):
        super().__init__(config)
        self.deberta = DebertaV2Model(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # Binary scam/safe head (2 classes, softmax at inference)
        self.binary_head = nn.Linear(config.hidden_size, 2)

        # Intent multi-label head (8 independent sigmoid outputs)
        self.intent_head = nn.Linear(config.hidden_size, 8)

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        intent_labels=None,
    ):
        outputs = self.deberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        # Use [CLS] token representation (first token)
        cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])

        binary_logits = self.binary_head(cls_output)   # [batch, 2]
        intent_logits = self.intent_head(cls_output)    # [batch, 8]

        loss = None
        if labels is not None:
            # Binary: CrossEntropyLoss (scam=1, safe=0)
            binary_loss = nn.CrossEntropyLoss()(binary_logits, labels)

            if intent_labels is not None:
                # Multi-label: BCEWithLogitsLoss (8 independent sigmoids)
                intent_loss = nn.BCEWithLogitsLoss()(intent_logits, intent_labels.float())
                # Combined loss: binary head weighted more heavily (hard gate)
                loss = 0.7 * binary_loss + 0.3 * intent_loss
            else:
                loss = binary_loss

        return {
            "loss": loss,
            "binary_logits": binary_logits,
            "intent_logits": intent_logits,
        }


# Load model from pretrained DeBERTa-v3-large
model = DualHeadDeBERTaTeacher.from_pretrained(MODEL_NAME)

# Enable gradient checkpointing if configured (required for T4)
if ACTIVE_CONFIG.get("gradient_checkpointing", False):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing: ENABLED")

# Print parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (FP32 est.): {total_params * 4 / 1024**3:.2f} GB")
print(f"Model size (FP16 est.): {total_params * 2 / 1024**3:.2f} GB")

In [ ]:
# =============================================================================
# Cell 5: Custom Trainer with Dual-Head Loss
# =============================================================================
# Subclass HuggingFace Trainer to handle dual-head model outputs and
# the custom intent_labels field in our dataset.

import numpy as np
import evaluate
from transformers import Trainer

# Load metrics
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")


def compute_metrics(eval_pred):
    """Compute binary F1, precision, recall from model predictions.

    The Trainer passes (predictions, label_ids) where predictions contains
    the binary_logits from our model's forward output.
    """
    logits, labels = eval_pred
    # logits is the binary_logits [N, 2]
    preds = np.argmax(logits, axis=-1)
    f1 = f1_metric.compute(predictions=preds, references=labels)["f1"]
    precision = precision_metric.compute(predictions=preds, references=labels)["precision"]
    recall = recall_metric.compute(predictions=preds, references=labels)["recall"]
    return {"f1": f1, "precision": precision, "recall": recall}


def dual_head_data_collator(features):
    """Custom data collator that properly handles intent_labels field.

    Standard collator does not handle the intent_labels tensor.
    This stacks all fields into proper batch tensors.
    """
    batch = {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "attention_mask": torch.stack([f["attention_mask"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
        "intent_labels": torch.stack([f["intent_labels"] for f in features]),
    }
    return batch


class DualHeadTrainer(Trainer):
    """Trainer subclass for the dual-head DeBERTa teacher model.

    Overrides compute_loss to:
    1. Extract intent_labels from inputs and pass to model
    2. Return the combined loss from the model's forward pass

    Also overrides prediction_step to return binary_logits for compute_metrics.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        intent_labels = inputs.pop("intent_labels")

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
            intent_labels=intent_labels,
        )

        loss = outputs["loss"]
        if return_outputs:
            # Return binary_logits as the prediction output for compute_metrics
            return loss, outputs["binary_logits"]
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        """Override to return binary_logits for metric computation."""
        inputs = self._prepare_inputs(inputs)
        labels = inputs.pop("labels")
        intent_labels = inputs.pop("intent_labels")

        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels=labels,
                intent_labels=intent_labels,
            )
            loss = outputs["loss"]

        if prediction_loss_only:
            return (loss, None, None)

        # Return binary_logits as predictions for compute_metrics
        return (loss, outputs["binary_logits"], labels)

print("DualHeadTrainer defined with custom compute_loss and prediction_step.")
print("Data collator handles intent_labels field.")

In [ ]:
# =============================================================================
# Cell 6: Checkpoint Resume Logic (D-02, D-04)
# =============================================================================
# Robust checkpointing that survives Colab session disconnects.
# All checkpoints saved to Google Drive. Training resumes from latest epoch.

import json
import glob
from datetime import datetime


def save_checkpoint(trainer, epoch, metrics, drive_dir):
    """Save full training state to Google Drive after each epoch.

    Args:
        trainer: The HuggingFace Trainer instance
        epoch: Completed epoch number (1-indexed)
        metrics: Dict of evaluation metrics for this epoch
        drive_dir: Google Drive checkpoint directory
    """
    checkpoint_dir = os.path.join(drive_dir, f"epoch_{epoch}")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save model weights
    trainer.model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)

    # Save optimizer and scheduler state
    torch.save(trainer.optimizer.state_dict(), os.path.join(checkpoint_dir, "optimizer.pt"))
    if trainer.lr_scheduler is not None:
        torch.save(trainer.lr_scheduler.state_dict(), os.path.join(checkpoint_dir, "scheduler.pt"))

    # Save training state metadata
    state = {
        "epoch": epoch,
        "metrics": metrics,
        "timestamp": datetime.now().isoformat(),
        "completed": True,
        "config": str(ACTIVE_CONFIG),
    }
    state_path = os.path.join(drive_dir, "training_state.json")
    with open(state_path, "w") as f:
        json.dump(state, f, indent=2)

    print(f"Checkpoint saved to {checkpoint_dir}")
    print(f"  Epoch: {epoch}, F1: {metrics.get('eval_f1', 'N/A')}")


def load_latest_checkpoint(drive_dir):
    """Resume from latest completed epoch checkpoint on Google Drive.

    Returns:
        checkpoint_path: Path to the latest checkpoint directory, or None
    """
    state_path = os.path.join(drive_dir, "training_state.json")
    if not os.path.exists(state_path):
        print("No previous checkpoint found. Starting fresh training.")
        return None

    with open(state_path) as f:
        state = json.load(f)

    epoch = state["epoch"]
    checkpoint_dir = os.path.join(drive_dir, f"epoch_{epoch}")

    if not os.path.exists(checkpoint_dir):
        print(f"WARNING: training_state.json references epoch {epoch} but directory missing.")
        # Try to find the latest existing epoch directory
        epoch_dirs = sorted(glob.glob(os.path.join(drive_dir, "epoch_*")))
        if epoch_dirs:
            checkpoint_dir = epoch_dirs[-1]
            print(f"Falling back to latest available: {checkpoint_dir}")
        else:
            print("No epoch directories found. Starting fresh training.")
            return None

    metrics = state.get("metrics", {})
    print(f"Resuming from epoch {epoch}")
    print(f"  Previous metrics: {metrics}")
    print(f"  Checkpoint: {checkpoint_dir}")
    return checkpoint_dir


# Check resume status
resume_checkpoint = load_latest_checkpoint(DRIVE_CHECKPOINT_DIR)

In [ ]:
# =============================================================================
# Cell 7: Training Execution
# =============================================================================
# Set up TrainingArguments from ACTIVE_CONFIG and run training.
# Trainer handles gradient accumulation, FP16, checkpointing, and resume.

import time
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=DRIVE_CHECKPOINT_DIR,
    # Evaluation and saving
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=None,              # Keep ALL checkpoints (D-04)
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    # Training hyperparameters from active config
    per_device_train_batch_size=ACTIVE_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=ACTIVE_CONFIG["per_device_train_batch_size"] * 2,
    gradient_accumulation_steps=ACTIVE_CONFIG["gradient_accumulation_steps"],
    learning_rate=ACTIVE_CONFIG["learning_rate"],
    num_train_epochs=ACTIVE_CONFIG["num_train_epochs"],
    warmup_ratio=ACTIVE_CONFIG["warmup_ratio"],
    weight_decay=ACTIVE_CONFIG["weight_decay"],
    lr_scheduler_type=ACTIVE_CONFIG["lr_scheduler_type"],
    max_grad_norm=ACTIVE_CONFIG["max_grad_norm"],
    # Memory optimization
    fp16=ACTIVE_CONFIG["fp16"],
    gradient_checkpointing=ACTIVE_CONFIG["gradient_checkpointing"],
    # Logging
    logging_steps=50,
    report_to="none",
    # Misc
    remove_unused_columns=False,         # Keep intent_labels in batch
    dataloader_num_workers=2,
)

# Create trainer with custom components
trainer = DualHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=dual_head_data_collator,
    compute_metrics=compute_metrics,
)

# Train (resume from checkpoint if one exists)
print("=" * 60)
print("STARTING TRAINING")
print(f"Config: {'T4' if ACTIVE_CONFIG is T4_CONFIG else 'A100' if ACTIVE_CONFIG is A100_CONFIG else 'RETRY_1' if ACTIVE_CONFIG is RETRY_1 else 'RETRY_2'}")
print(f"Effective batch size: {ACTIVE_CONFIG['per_device_train_batch_size'] * ACTIVE_CONFIG['gradient_accumulation_steps']}")
print(f"Epochs: {ACTIVE_CONFIG['num_train_epochs']}")
print(f"Learning rate: {ACTIVE_CONFIG['learning_rate']}")
print("=" * 60)

train_start = time.time()

if resume_checkpoint is not None:
    print(f"Resuming from checkpoint: {resume_checkpoint}")
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
else:
    print("Starting fresh training...")
    train_result = trainer.train()

train_elapsed = time.time() - train_start
print(f"\nTraining complete in {train_elapsed/60:.1f} minutes")
print(f"Training loss: {train_result.training_loss:.4f}")

# Save best model to dedicated directory on Drive
best_dir = os.path.join(DRIVE_CHECKPOINT_DIR, "best_binary")
os.makedirs(best_dir, exist_ok=True)
trainer.model.save_pretrained(best_dir)
tokenizer.save_pretrained(best_dir)
print(f"Best model saved to {best_dir}")

In [ ]:
# =============================================================================
# Cell 8: Synthetic Test Evaluation (D-07, TEXT-04-T1)
# =============================================================================
# Internal quality bar: F1 > 0.95 on synthetic test set.
# If this passes but holdout fails, it indicates a generalization problem
# (teacher learned LLM output style, not real scam patterns -- Pitfall 1).

from sklearn.metrics import classification_report

print("=" * 60)
print("SYNTHETIC TEST EVALUATION")
print("=" * 60)

# Run predictions on synthetic test set
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels = test_results.label_ids

# Compute metrics
synthetic_f1 = f1_metric.compute(predictions=test_preds, references=test_labels)["f1"]
synthetic_precision = precision_metric.compute(predictions=test_preds, references=test_labels)["precision"]
synthetic_recall = recall_metric.compute(predictions=test_preds, references=test_labels)["recall"]

print(f"\nSynthetic Test Results:")
print(f"  F1:        {synthetic_f1:.4f}")
print(f"  Precision: {synthetic_precision:.4f}")
print(f"  Recall:    {synthetic_recall:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=["safe", "scam"]))

# Gate check
if synthetic_f1 > 0.95:
    print(f"SYNTHETIC TEST GATE: PASSED (F1={synthetic_f1:.4f} > 0.95)")
else:
    print(f"SYNTHETIC TEST GATE: FAILED (F1={synthetic_f1:.4f} <= 0.95)")
    print("WARNING: Teacher may not have converged. Check training loss curve.")
    print("Consider running more epochs or adjusting learning rate.")

assert synthetic_f1 > 0.95, (
    f"Synthetic test F1 = {synthetic_f1:.4f}, expected > 0.95. "
    f"Teacher has not converged on synthetic data. "
    f"Check training loss and consider more epochs or lower learning rate."
)

In [ ]:
# =============================================================================
# Cell 9: Holdout Evaluation + Per-Vector Breakdown (D-06, D-14, TEXT-04-T2)
# =============================================================================
# Hard gate: Binary F1 > 0.80 on real-world holdout.
# Per-vector breakdown identifies which scam vectors the teacher struggles with,
# informing whether failure is a training config issue or data quality issue.

import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

print("=" * 60)
print("HOLDOUT EVALUATION (HARD GATE: F1 > 0.80)")
print("=" * 60)

# Run predictions on holdout
holdout_results = trainer.predict(holdout_dataset)
holdout_preds = np.argmax(holdout_results.predictions, axis=-1)
holdout_labels = holdout_results.label_ids

# Overall binary metrics
holdout_f1 = f1_score(holdout_labels, holdout_preds)
holdout_precision = precision_score(holdout_labels, holdout_preds)
holdout_recall = recall_score(holdout_labels, holdout_preds)

print(f"\nOverall Holdout Results:")
print(f"  F1:        {holdout_f1:.4f}")
print(f"  Precision: {holdout_precision:.4f}")
print(f"  Recall:    {holdout_recall:.4f}")

print("\nClassification Report:")
print(classification_report(holdout_labels, holdout_preds, target_names=["safe", "scam"]))

# Per-vector breakdown (D-14)
print("\nPer-Vector Holdout Breakdown:")
print("-" * 70)

vector_data = []
for i, sample in enumerate(holdout_dataset.samples):
    vector_data.append({
        "vector": sample["vector"],
        "true_label": holdout_labels[i],
        "pred_label": holdout_preds[i],
    })

df = pd.DataFrame(vector_data)
per_vector_results = []

for vector in sorted(df["vector"].unique()):
    mask = df["vector"] == vector
    v_true = df.loc[mask, "true_label"].values
    v_pred = df.loc[mask, "pred_label"].values
    n = len(v_true)

    # For vectors with only one class, F1 is not meaningful in the binary sense
    if len(set(v_true)) == 1:
        # All same label -- compute accuracy instead
        accuracy = (v_true == v_pred).mean()
        per_vector_results.append({
            "Vector": vector,
            "N": n,
            "F1": accuracy,  # use accuracy as proxy for single-class
            "Precision": accuracy,
            "Recall": accuracy,
            "Note": "*" if n < 10 else "",
        })
    else:
        v_f1 = f1_score(v_true, v_pred, zero_division=0)
        v_prec = precision_score(v_true, v_pred, zero_division=0)
        v_rec = recall_score(v_true, v_pred, zero_division=0)
        per_vector_results.append({
            "Vector": vector,
            "N": n,
            "F1": v_f1,
            "Precision": v_prec,
            "Recall": v_rec,
            "Note": "*" if n < 10 else "",
        })

results_df = pd.DataFrame(per_vector_results)
results_df = results_df.sort_values("F1", ascending=True)  # worst vectors first
print(results_df.to_string(index=False))
print("\n* Vectors with <10 samples -- per-vector metrics may be unreliable")

# Intent head metrics (informational only per D-06)
print("\n" + "=" * 60)
print("INTENT HEAD METRICS (informational, does NOT block Phase 4)")
print("=" * 60)

# Get intent predictions from the model on holdout
device = next(model.parameters()).device
model.eval()
all_intent_preds = []
all_intent_labels = []

holdout_loader = DataLoader(holdout_dataset, batch_size=32, shuffle=False,
                            collate_fn=dual_head_data_collator)
with torch.no_grad():
    for batch in holdout_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        intent_preds = (torch.sigmoid(outputs["intent_logits"]) > 0.5).int().cpu()
        all_intent_preds.append(intent_preds)
        all_intent_labels.append(batch["intent_labels"].int())

all_intent_preds = torch.cat(all_intent_preds, dim=0).numpy()
all_intent_labels = torch.cat(all_intent_labels, dim=0).numpy()

for i, label_name in enumerate(INTENT_LABELS):
    true = all_intent_labels[:, i]
    pred = all_intent_preds[:, i]
    if true.sum() == 0:
        print(f"  {label_name:25s}  -- no positive samples in holdout")
        continue
    i_prec = precision_score(true, pred, zero_division=0)
    i_rec = recall_score(true, pred, zero_division=0)
    i_f1 = f1_score(true, pred, zero_division=0)
    print(f"  {label_name:25s}  F1={i_f1:.3f}  Prec={i_prec:.3f}  Rec={i_rec:.3f}  (n={int(true.sum())})")

print("\nNote: Intent metrics are informational only. Holdout sample count (202)")
print("is too small for meaningful per-label intent metrics (D-06).")

# Gate check
print("\n" + "=" * 60)
if holdout_f1 > 0.80:
    print(f"HOLDOUT GATE: PASSED (F1={holdout_f1:.4f} > 0.80)")
    print("Teacher meets the Phase 3 hard gate. Ready for soft label generation.")
else:
    print(f"HOLDOUT GATE: FAILED (F1={holdout_f1:.4f} <= 0.80)")
    print("See per-vector breakdown above to diagnose failure mode.")
    print("Next steps:")
    print("  1. If F1 < 0.80: Try RETRY_1 config (lower LR, more epochs)")
    print("  2. If RETRY_1 also fails: Try RETRY_2 (class weighting)")
    print("  3. If both retries fail: Escalate to data augmentation (D-13)")
print("=" * 60)

assert holdout_f1 > 0.80, (
    f"Holdout F1 = {holdout_f1:.4f}, expected > 0.80. "
    f"Teacher does not generalize to real-world scam patterns. "
    f"Review per-vector breakdown above and try RETRY_1 or RETRY_2 config."
)